In [138]:
import numpy as np
import sklearn
import torch
import os

In [139]:
if not os.path.exists('tree_species_classifier_data.npz'):
  !wget -O tree_species_classifier_data.npz "https://www.dropbox.com/scl/fi/b7mw23k3ifaeui9m8nnn3/tree_species_classifier_data.npz?rlkey=bgxp37c1t04i7q35waf3slc26&dl=1"

In [140]:
data = np.load('tree_species_classifier_data.npz')
train_features = data['train_features']
train_labels = data['train_labels']
test_features = data['test_features']
test_labels = data['test_labels']

### Data Exploration

In [141]:
matrices = [train_features, train_labels, test_features, test_labels]
for matrix in matrices:
    print("Type:", type(matrix))
    print("Shape:", matrix.shape)
    print("Range:", np.ptp(matrix)) 
    print("")


Type: <class 'numpy.ndarray'>
Shape: (15707, 426)
Range: 14998

Type: <class 'numpy.ndarray'>
Shape: (15707,)
Range: 7

Type: <class 'numpy.ndarray'>
Shape: (1554, 426)
Range: 6908

Type: <class 'numpy.ndarray'>
Shape: (1554,)
Range: 7



### Pre-Process Data

In [142]:
from sklearn.decomposition import PCA

pca = PCA(n_components=32, whiten=True).fit(train_features)
train_features_compressed = pca.transform(train_features)
test_features_compressed = pca.transform(test_features)
train_features_compressed.shape

(15707, 32)

### Classifiers using scikit-learn

In [143]:
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

linear_model = LogisticRegression(random_state=123).fit(train_features_compressed, train_labels)
nn_model = MLPClassifier(random_state=123, hidden_layer_sizes=(100, 100, 100)).fit(train_features_compressed, train_labels)

In [144]:
linear_model.score(test_features_compressed, test_labels)

0.833976833976834

In [145]:
nn_model.score(test_features_compressed, test_labels)

0.8371943371943372

### Classifiers using PyTorch

In [146]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

In [147]:
train_dataset = TensorDataset(
    torch.tensor(train_features_compressed, dtype=torch.float),
    torch.tensor(train_labels, dtype=torch.long)
)
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)

test_dataset = TensorDataset(
    torch.tensor(test_features_compressed, dtype=torch.float),
    torch.tensor(test_labels, dtype=torch.long)
)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [148]:
def model_accuracy(model, dataloader):
    model.eval()
    total = 0
    correct = 0

    with torch.no_grad():
        for features, labels in dataloader:
            outputs = model(features)
            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    return float(correct / total)

In [149]:
def train_model(model, dataloader):
    optim = torch.optim.SGD(model.parameters(), lr=1e-2, weight_decay=1e-3)
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(100):
        for features, labels in dataloader:
            optim.zero_grad()
            outputs = model(features)
            loss = loss_fn(outputs, labels)
            loss.backward()
            optim.step()

        print(f"Epoch {epoch}: Accuracy = {model_accuracy(model, dataloader)}")

In [150]:
pt_linear_model = nn.Sequential(
    nn.Linear(32,8), #32 inputs, 8 outputs
)

train_model(pt_linear_model, train_dataloader)
print(f"PyTorch Linear Model accuracy on test set: {model_accuracy(pt_linear_model, test_dataloader)}")

Epoch 0: Accuracy = 0.7656458903673521
Epoch 1: Accuracy = 0.7937225440886229
Epoch 2: Accuracy = 0.8070923791939899
Epoch 3: Accuracy = 0.8176609155153753
Epoch 4: Accuracy = 0.8237728401349717
Epoch 5: Accuracy = 0.8278474565480359
Epoch 6: Accuracy = 0.8298847647545681
Epoch 7: Accuracy = 0.8316037435538295
Epoch 8: Accuracy = 0.832877061182912
Epoch 9: Accuracy = 0.8337047176418158
Epoch 10: Accuracy = 0.8346597058636277
Epoch 11: Accuracy = 0.8359330234927103
Epoch 12: Accuracy = 0.8369516775959763
Epoch 13: Accuracy = 0.8374610046476093
Epoch 14: Accuracy = 0.8388616540396002
Epoch 15: Accuracy = 0.8394983128541414
Epoch 16: Accuracy = 0.8400713057872287
Epoch 17: Accuracy = 0.8403259693130452
Epoch 18: Accuracy = 0.8401986375501369
Epoch 19: Accuracy = 0.8405806328388616
Epoch 20: Accuracy = 0.8411536257719489
Epoch 21: Accuracy = 0.8412809575348571
Epoch 22: Accuracy = 0.8420449481123066
Epoch 23: Accuracy = 0.8421722798752149
Epoch 24: Accuracy = 0.8430636022155726
Epoch 25: A

In [151]:
hidden_width = 100

# neural network with 3 hidden layers of size 100
pt_nn_model = nn.Sequential(
    nn.Linear(32, hidden_width),
    nn.ReLU(),
    nn.Linear(hidden_width, hidden_width),
    nn.ReLU(),
    nn.Linear(hidden_width, hidden_width),
    nn.ReLU(),
    nn.Linear(hidden_width, hidden_width),
    nn.ReLU(),
    nn.Linear(hidden_width, 8)
)

train_model(pt_nn_model, train_dataloader)
print(f"PyTorch NN Model accuracy on test set: {model_accuracy(pt_nn_model, test_dataloader)}")

Epoch 0: Accuracy = 0.2534538740688865
Epoch 1: Accuracy = 0.33023492710256575
Epoch 2: Accuracy = 0.4024320366715477
Epoch 3: Accuracy = 0.6169860571719615
Epoch 4: Accuracy = 0.7299929967530401
Epoch 5: Accuracy = 0.7853186477366779
Epoch 6: Accuracy = 0.814541287324123
Epoch 7: Accuracy = 0.8262558095116826
Epoch 8: Accuracy = 0.8386706563952377
Epoch 9: Accuracy = 0.8494938562424397
Epoch 10: Accuracy = 0.8578340867129305
Epoch 11: Accuracy = 0.8663016489463297
Epoch 12: Accuracy = 0.8708219265295728
Epoch 13: Accuracy = 0.8769975170306233
Epoch 14: Accuracy = 0.8815177946138665
Epoch 15: Accuracy = 0.8857197427898389
Epoch 16: Accuracy = 0.8878843827592793
Epoch 17: Accuracy = 0.8892850321512701
Epoch 18: Accuracy = 0.8890940345069077
Epoch 19: Accuracy = 0.8910040109505316
Epoch 20: Accuracy = 0.8962246132297702
Epoch 21: Accuracy = 0.9023365378493665
Epoch 22: Accuracy = 0.9033551919526326
Epoch 23: Accuracy = 0.9034825237155408
Epoch 24: Accuracy = 0.9043101801744445
Epoch 25: 